# 02 - Data Validation

**Objective:** Validate breast cancer records using Pydantic models and Pandera DataFrame schemas.

**Tools:** Pydantic (row-level), Pandera (DataFrame-level)

**Steps:**
1. Define a Pydantic model for breast cancer records
2. Implement row-level validation
3. Define a Pandera DataFrame schema
4. Implement DataFrame-level validation
5. Test on clean and corrupted data

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from pydantic import BaseModel, field_validator
import pandera as pa
from pandera.typing import DataFrame, Series

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
PROCESSED_DIR = Path("../data/processed")
CORRUPED_DIR = Processed_DIR / "corrupted"
df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")

print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Data shape: (569, 31)
Columns: ['Diagnosis', 'Mean_Radius', 'SE_Radius', 'Worst_Radius', 'Mean_Texture', 'SE_Texture', 'Worst_Texture', 'Mean_Perimeter', 'SE_Perimeter', 'Worst_Perimeter', 'Mean_Area', 'SE_Area', 'Worst_Area', 'Mean_Smoothness', 'SE_Smoothness', 'Worst_Smoothness', 'Mean_Compactness', 'SE_Compactness', 'Worst_Compactness', 'Mean_Concavity', 'SE_Concavity', 'Worst_Concavity', 'Mean_ConcavePoints', 'SE_ConcavePoints', 'Worst_ConcavePoints', 'Mean_Symmetry', 'SE_Symmetry', 'Worst_Symmetry', 'Mean_FractalDimension', 'SE_FractalDimension', 'Worst_FractalDimension']


### Pydantic Model Definition

Pydantic validates data at the **record level** — each row is checked individually.
Define a model with typed fields and add `@field_validator` decorators to enforce
business rules like valid diagnosis values, positive measurements, etc.

This catches row-level issues like a Diagnosis of 2 or a negative radius
that might slip through type-only checks.

In [4]:
# === Executed Example: Pydantic Row-Level Validation ===
# This cell uses a small inline dataset (no CSV needed) to demonstrate
# how Pydantic catches invalid rows at the record level.

import pandas as pd
from pydantic import BaseModel, field_validator

data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "diagnosis": [0, 1, 2, 0, 1],
    "radius": [14.5, 18.2, -3.0, 8.0, 999.0],
})

class Record(BaseModel):
    id: int
    diagnosis: int
    radius: float

    @field_validator("diagnosis")
    @classmethod
    def diagnosis_binary(cls, v):
        if v not in [0, 1]:
            raise ValueError(f"Diagnosis must be 0 or 1, got {v}")
        return v

    @field_validator("radius")
    @classmethod
    def radius_positive(cls, v):
        if not (0 < v <= 50):
            raise ValueError(f"Radius out of range: {v}")
        return v

valid = []
for _, row in data.iterrows():
    try:
        Record(**row.to_dict())
        valid.append(True)
    except ValueError:
        valid.append(False)

print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

Pydantic: 3/5 rows valid


In [5]:
# === Commented Template: Pydantic Row-Level Validation ===
# Copy, paste, and adapt to your own dataset. Uncomment to use.

# import pandas as pd
# from pydantic import BaseModel, field_validator

# data = pd.DataFrame({
#     "field_1": [val1, val2, val3],
#     "field_2": [val1, val2, val3],
# })

# class MyRecord(BaseModel):
#     field_1: int
#     field_2: float

#     @field_validator("field_1")
#     @classmethod
#     def rule_name(cls, v):
#         if not (lower <= v <= upper):
#             raise ValueError(f"Constraint description: {v}")
#         return v

# valid = []
# for _, row in data.iterrows():
#     try:
#         MyRecord(**row.to_dict())
#         valid.append(True)
#     except ValueError:
#         valid.append(False)

# print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

In [6]:
class BreastCancerRecord(BaseModel):
    """Pydantic model for a single breast cancer record."""

    Diagnosis: int
    Mean_Radius: float
    SE_Radius: float
    Worst_Radius: float
    Mean_Texture: float
    SE_Texture: float
    Worst_Texture: float
    Mean_Perimeter: float
    SE_Perimeter: float
    Worst_Perimeter: float
    Mean_Area: float
    SE_Area: float
    Worst_Area: float
    Mean_Smoothness: float
    SE_Smoothness: float
    Worst_Smoothness: float
    Mean_Compactness: float
    SE_Compactness: float
    Worst_Compactness: float
    Mean_Concavity: float
    SE_Concavity: float
    Worst_Concavity: float
    Mean_ConcavePoints: float
    SE_ConcavePoints: float
    Worst_ConcavePoints: float
    Mean_Symmetry: float
    SE_Symmetry: float
    Worst_Symmetry: float
    Mean_FractalDimension: float
    SE_FractalDimension: float
    Worst_FractalDimension: float

    @field_validator("Diagnosis")
    @classmethod
    def diagnosis_binary(cls, v):
        if v not in [0, 1]:
            raise ValueError(f"Diagnosis must be 0 or 1, got {v}")
        return v

    @field_validator("Mean_Radius", "SE_Radius", "Worst_Radius")
    @classmethod
    def radius_positive(cls, v):
        if v < 0:
            raise ValueError(f"Radius must be positive: {v}")
        return v

    @field_validator("Mean_Texture", "SE_Texture", "Worst_Texture")
    @classmethod
    def texture_positive(cls, v):
        if v < 0:
            raise ValueError(f"Texture must be positive: {v}")
        return v

    @field_validator("Mean_Perimeter", "SE_Perimeter", "Worst_Perimeter")
    @classmethod
    def perimeter_positive(cls, v):
        if v < 0:
            raise ValueError(f"Perimeter must be positive: {v}")
        return v

    @field_validator("Mean_Area", "SE_Area", "Worst_Area")
    @classmethod
    def area_positive(cls, v):
        if v < 0:
            raise ValueError(f"Area must be positive: {v}")
        return v

    @field_validator(
        "Mean_Smoothness", "SE_Smoothness", "Worst_Smoothness",
        "Mean_Compactness", "SE_Compactness", "Worst_Compactness",
        "Mean_Concavity", "SE_Concavity", "Worst_Concavity",
        "Mean_ConcavePoints", "SE_ConcavePoints", "Worst_ConcavePoints",
        "Mean_Symmetry", "SE_Symmetry", "Worst_Symmetry",
    )
    @classmethod
    def non_negative(cls, v):
        if v < 0:
            raise ValueError(f"Feature cannot be negative: {v}")
        return v

    @field_validator("Mean_FractalDimension", "SE_FractalDimension", "Worst_FractalDimension")
    @classmethod
    def fractal_in_range(cls, v):
        if not (0 <= v <= 1):
            raise ValueError(f"Fractal dimension must be between 0 and 1, got {v}")
        return v


In [7]:
def validate_with_pydantic(df):
    """Validate each row of the DataFrame using the Pydantic model.

    Returns a list of booleans where True means the row is valid.
    """
    valid = []
    for _, row in df.iterrows():
        try:
            BreastCancerRecord(**row.to_dict())
            valid.append(True)
        except Exception:
            valid.append(False)

    valid_count = sum(valid)
    invalid_count = len(valid) - valid_count
    print(f"Pydantic: {valid_count} valid, {invalid_count} invalid out of {len(valid)} rows")
    return valid


valid_flags = validate_with_pydantic(df)
print(f"Pass rate: {sum(valid_flags) / len(valid_flags):.1%}")

Pydantic: 569 valid, 0 invalid out of 569 rows
Pass rate: 100.0%


### Pandera Schema Definition

Pandera validates data at the **DataFrame level** — instead of checking one row at a time,
it defines column-wide constraints (data types, value ranges, nullable rules, etc.).

This is more efficient for bulk validation and catches issues like a column
that unexpectedly contains strings instead of numbers.

In [13]:
# === Executed Example: Pandera DataFrame-Level Validation ===
# This cell uses the same inline dataset to demonstrate how Pandera
# validates all rows at once, collecting all failures.

import pandas as pd
import pandera.pandas as pa

data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "diagnosis": [0, 1, 2, 0, 1],
    "radius": [14.5, 18.2, -3.0, 8.0, 999.0],
})

Schema = pa.DataFrameSchema({
    "id": pa.Column(pa.Int, nullable=False),
    "diagnosis": pa.Column(pa.Int, checks=pa.Check.isin([0, 1])),
    "radius": pa.Column(pa.Float, checks=[pa.Check.greater_than(0), pa.Check.le(50)]),
})

try:
    Schema.validate(data, lazy=True)
    print("Pandera: all rows passed")
except pa.errors.SchemaErrors as e:
    print(f"Pandera: {len(e.failure_cases)} failures found")
    print(e.failure_cases[["index", "column", "check", "failure_case"]].head())

Pandera: 3 failures found
   index     column                      check  failure_case
0      2  diagnosis               isin([0, 1])           2.0
1      2     radius            greater_than(0)          -3.0
2      4     radius  less_than_or_equal_to(50)         999.0


In [9]:
# === Commented Template: Pandera DataFrame-Level Validation ===
# Copy, paste, and adapt to your own dataset. Uncomment to use.

# import pandas as pd
# import pandera as pa

# data = pd.DataFrame({
#     "field_1": [val1, val2, val3],
#     "field_2": [val1, val2, val3],
# })

# MySchema = pa.DataFrameSchema({
#     "field_1": pa.Column(pa.Int, nullable=False),
#     "field_2": pa.Column(pa.Float, checks=[pa.Check.ge(min_val), pa.Check.le(max_val)]),
# })

# try:
#     MySchema.validate(data, lazy=True)
#     print("Pandera: all rows valid")
# except pa.errors.SchemaErrors as e:
#     print(f"Pandera: {len(e.failure_cases)} failure(s)")

In [10]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

feature_cols = [
    "Mean_Radius", "SE_Radius", "Worst_Radius",
    "Mean_Texture", "SE_Texture", "Worst_Texture",
    "Mean_Perimeter", "SE_Perimeter", "Worst_Perimeter",
    "Mean_Area", "SE_Area", "Worst_Area",
    "Mean_Smoothness", "SE_Smoothness", "Worst_Smoothness",
    "Mean_Compactness", "SE_Compactness", "Worst_Compactness",
    "Mean_Concavity", "SE_Concavity", "Worst_Concavity",
    "Mean_ConcavePoints", "SE_ConcavePoints", "Worst_ConcavePoints",
    "Mean_Symmetry", "SE_Symmetry", "Worst_Symmetry",
    "Mean_FractalDimension", "SE_FractalDimension", "Worst_FractalDimension",
]

BreastCancerSchema = pa.DataFrameSchema({
    "Diagnosis": pa.Column(pa.Int, checks=pa.Check.isin([0, 1])),
    **{
        col: pa.Column(pa.Float, checks=pa.Check.ge(0))
        for col in feature_cols
    },
})

print("BreastCancerSchema defined with all 31 columns")

BreastCancerSchema defined with all 31 columns


In [11]:
def validate_with_pandera(df):
    """Validate the entire DataFrame using the Pandera schema."""
    try:
        BreastCancerSchema.validate(df, lazy=True)
        print("Pandera: all rows passed validation")
    except pa.errors.SchemaErrors as e:
        print(f"Pandera: {len(e.failure_cases)} failure(s) found")
        print(e.failure_cases[["index", "column", "check", "failure_case"]].head())


validate_with_pandera(df)

Pandera: all rows passed validation


In [12]:
# Load corrupted data with missing values in Diagnosis
df_corrupted = pd.read_csv(PROCESSED_DIR / "corrupted" / "corrupted_missing_light.csv")
print(f"Corrupted data shape: {df_corrupted.shape}")
print(f"Missing values per column:\n{df_corrupted.isnull().sum()}\n")

# Pydantic validation on corrupted data
print("--- Pydantic Validation ---")
valid_corrupted = validate_with_pydantic(df_corrupted)
cv = sum(valid_corrupted)
ct = len(valid_corrupted)
print(f"Corrupted pass rate: {cv}/{ct} ({cv/ct:.1%})")

# Pandera validation on corrupted data
print("\n--- Pandera Validation ---")
validate_with_pandera(df_corrupted)

print("\n" + "="*60 + "\n")

# Generate corrupted data programmatically using the injection module
# Load different corrupted variants from pre-generated files
df_outliers = pd.read_csv(PROCESSED_DIR / "corrupted" / "corrupted_outliers.csv")

print(f"Outlier data shape: {df_outliers.shape}")
print(f"Mean_Radius range: [{df_outliers['Mean_Radius'].min():.2f}, {df_outliers['Mean_Radius'].max():.2f}]\n")

print("--- Pydantic Validation (outliers) ---")
valid_outliers = validate_with_pydantic(df_outliers)
print(f"Pass rate: {sum(valid_outliers)}/{len(valid_outliers)} ({sum(valid_outliers)/len(valid_outliers):.1%})")

print("\n--- Pandera Validation (outliers) ---")
validate_with_pandera(df_outliers)

Corrupted data shape: (569, 31)
Missing values per column:
Diagnosis                 28
Mean_Radius                0
SE_Radius                  0
Worst_Radius               0
Mean_Texture               0
SE_Texture                 0
Worst_Texture              0
Mean_Perimeter             0
SE_Perimeter               0
Worst_Perimeter            0
Mean_Area                  0
SE_Area                    0
Worst_Area                 0
Mean_Smoothness            0
SE_Smoothness              0
Worst_Smoothness           0
Mean_Compactness           0
SE_Compactness             0
Worst_Compactness          0
Mean_Concavity             0
SE_Concavity               0
Worst_Concavity            0
Mean_ConcavePoints         0
SE_ConcavePoints           0
Worst_ConcavePoints        0
Mean_Symmetry              0
SE_Symmetry                0
Worst_Symmetry             0
Mean_FractalDimension      0
SE_FractalDimension        0
Worst_FractalDimension     0
dtype: int64

--- Pydantic Validation ---


### Exercises

1. **Add more validators**: Add a `@field_validator` for `Mean_Radius` that rejects values below 0 or above 30 (a realistic upper bound for breast cancer data).
2. **Compare error messages**: Run the same corrupted data through both Pydantic and Pandera — which one gives more informative error messages?
3. **Performance test**: Time both validators on the full dataset (use `%%timeit` in a cell) — is Pandera noticeably faster for bulk validation?
4. **Custom error handler**: Modify `validate_with_pydantic()` to collect and return the actual error messages (not just True/False) so students can see why a row failed.
5. **Schema evolution**: What happens if a new column is added to the CSV? Will Pandera reject it or pass it through? Check the `strict` parameter in `pa.DataFrameSchema`.